In [ ]:
import os, glob, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve,recall_score,precision_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from xgboost.callback import EarlyStopping as XGBEarlyStopping

In [ ]:

import os
from pathlib import Path

BASE_PATH = os.path.join(os.getcwd(), 'fraud_stream_parquet')
OUT_DIR = os.path.join(os.getcwd(), 'baseline1_results')


os.makedirs(OUT_DIR, exist_ok=True)

FEATURES = [
    'TX_AMOUNT', 'TX_TIME_DAYS', 'TX_TIME_SECONDS',
    'x_customer_id','y_customer_id','mean_amount','std_amount','mean_nb_tx_per_day',
    'x_terminal_id','y_terminal_id'
]
TARGET = 'TX_FRAUD'

PRE_MONTHS = 4               
VAL_DAYS_LAST_MONTH = 14       
START_DATE  = "2025-01-01" 
GRANULARITY = 'week'         
TIMELINE_FILE = os.path.join(BASE_PATH, 'timeline.parquet')  


def compute_base_week(eval_df: pd.DataFrame) -> int:
    base_week = int(eval_df['TX_TIME_DAYS'].min() // 7) + 1
    base_day = int(eval_df['TX_TIME_DAYS'].min())  
    base_date = pd.to_datetime(START_DATE) + pd.Timedelta(days=base_day)
    print(f"Semana base: {base_week}, Fecha: {base_date.strftime('%Y-%m-%d')}")
    return base_week


In [ ]:
def load_all_parquet_by_year(base_path: str) -> pd.DataFrame:
    year_dirs = sorted(glob.glob(os.path.join(base_path, "TX_YEAR=*")))
    dfs = []
    for ydir in year_dirs:
        files = sorted(glob.glob(os.path.join(ydir, "*.parquet")))
        if not files:
            continue
        dfs.append(pd.concat([pd.read_parquet(f) for f in files], ignore_index=True))
    if not dfs:
        raise RuntimeError(f"No se encontraron Parquet en {base_path}")
    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values(['TX_YEAR','TX_MONTH','TX_DAY','TX_TIME_SECONDS'], kind='mergesort').reset_index(drop=True)
    #print(df.head())
   # print(df.columns.tolist())
    return df


def add_time_indexes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    first_year = int(df['TX_YEAR'].min())
    df['_month_idx'] = (df['TX_YEAR'] - first_year) * 12 + df['TX_MONTH']  # 1..24

    df['_week_idx']  = (df['TX_TIME_DAYS'] // 7).astype(int) + 1  
    df['_day_abs'] = (df['_month_idx'] - 1) * 31 + df['TX_DAY']  
    return df

def group_chunks(df: pd.DataFrame, granularity='month', start_date_str=START_DATE):
    if granularity == 'month':
        for (y, m), g in df.groupby(['TX_YEAR','TX_MONTH'], sort=True):
            yield (int(y), int(m)), g, f"{int(y)}-{int(m):02d}"

    elif granularity == 'day':
        for (y, m, d), g in df.groupby(['TX_YEAR','TX_MONTH','TX_DAY'], sort=True):
            yield (int(y), int(m), int(d)), g, f"{int(y)}-{int(m):02d}-{int(d):02d}"

    elif granularity == 'week':
        d0 = pd.to_datetime(start_date_str)
        for wk, g in df.groupby('_week_idx', sort=True):
            wk = int(wk)
            wstart = d0 + pd.Timedelta(days=(wk-1)*7)
            iso = wstart.isocalendar()  # (year, week, weekday)
            label = f"{int(iso.year)}-W{int(iso.week):02d}"
            yield wk, g, label
    else:
        raise ValueError("granularity debe ser 'month' | 'week' | 'day'.")


In [ ]:

class NoOpScaler:
    def fit(self, X, y=None): return self
    def transform(self, X):    return X


def make_model(model_name: str, scale_pos_weight: float = 1.0, random_state: int = 42):
    name = model_name.lower()
    if name in ("xgb", "xgboost"):
        return XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1,
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',   # o 'aucpr'
            tree_method='hist',
            verbosity=0
        )
    elif name in ("rf","random_forest","randomforest"):
        return RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=random_state
        )
    elif name in ("svm","svc"):
        # probability=True para tener predict_proba
        return LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=random_state,
            dual="auto"
        )
    elif name in ("lr","logreg","logistic","logistic_regression"):
        return LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            class_weight="balanced",
            n_jobs=2,
            random_state=random_state
        )
    else:
        raise ValueError(f"Modelo no soportado: {model_name}")


def get_scores(clf, X):

    if hasattr(clf, "predict_proba"):
        s = clf.predict_proba(X)[:, 1]
        return s.astype("float64"), "proba"

    if hasattr(clf, "decision_function"): 
        s = clf.decision_function(X)
        return s.astype("float64"), "decision"

    s = clf.predict(X).astype("float64")
    return s, "label"


def fit_model(train_df, val_df, features, model_name, target='TX_FRAUD', random_state=42):
   
    Xtr = train_df[features].astype('float32').values
    ytr = train_df[target].astype('int32').values
    Xv  = val_df[features].astype('float32').values
    yv  = val_df[target].astype('int32').values


    pos = int(ytr.sum())
    neg = int(len(ytr) - pos)
    scale_pos_weight = float(neg / max(1, pos))

  
    name = model_name.lower()
    if name in ("svm","svc","lr","logreg","logistic","logistic_regression"):
        scaler = StandardScaler().fit(Xtr)
        Xtr_s  = scaler.transform(Xtr)
        Xv_s   = scaler.transform(Xv)
    else:
        scaler = NoOpScaler()
        Xtr_s, Xv_s = Xtr, Xv

    clf = make_model(model_name, scale_pos_weight=scale_pos_weight, random_state=random_state)


    t0 = time.perf_counter()
    clf.fit(Xtr_s, ytr)
    train_time_s = time.perf_counter() - t0

 
    t0 = time.perf_counter()
    pv, score_mode = get_scores(clf, Xv_s)
    inf_time_val_s = time.perf_counter() - t0

 
    try:
        ap_ref = float(average_precision_score(yv, pv)) if yv.sum() > 0 else float('nan')
    except Exception:
        ap_ref = float('nan')

    prec, rec, thr = precision_recall_curve(yv, pv)
    f1s = (2 * prec * rec) / (prec + rec + 1e-12)
    if len(thr) > 0:
        best_idx = int(np.nanargmax(f1s[:-1]))
        best_thr = float(thr[best_idx])
        f1_ref   = float(f1s[best_idx])
    else:
        best_thr = 0.5 if score_mode == "proba" else 0.0
        f1_ref   = float(f1_score(yv, (pv >= best_thr).astype(int), zero_division=0))


    ms_per_tx_val = (inf_time_val_s / max(1, len(yv))) * 1e3

    return scaler, clf, best_thr, ap_ref, f1_ref, train_time_s, ms_per_tx_val


def evaluate_stream(df, features, scaler, clf, thr, granularity='week', base_week=None):
    import pandas as pd

    rows = []
    tot_inf_s = 0.0
    tot_n     = 0

    for key, chunk, label in group_chunks(df, granularity=granularity, start_date_str=START_DATE):

        if isinstance(key, int) and base_week is not None:
            pos = int(key) - int(base_week) + 1
        else:
            pos = int(key) if isinstance(key, int) else None

        X = scaler.transform(chunk[features].astype('float32').values)
        y = chunk[TARGET].astype('int32').values
        n = int(len(y))

        # Scores + tiempo
        t0 = time.perf_counter()
        s, _mode = get_scores(clf, X)
        dt_s = time.perf_counter() - t0

        tot_inf_s += dt_s
        tot_n     += n

        # Métricas
        try:
            ap = float(average_precision_score(y, s)) if y.sum() > 0 else float('nan')
        except Exception:
            ap = float('nan')

        f1 = float(f1_score(y, (s >= thr).astype(int), zero_division=0))

        # Latencias
        infer_total_ms_ch  = dt_s * 1e3

        rows.append({
            "pos": pos, "label": label, "n": n,
            "AUPRC": ap, "F1": f1,
            "infer_total_ms_chunk": infer_total_ms_ch
        })

    import pandas as pd
    metrics = pd.DataFrame(rows).sort_values('pos').reset_index(drop=True)
    return metrics


In [29]:
def split_pretrain_val(df: pd.DataFrame, pre_months=4, val_days_last_month=14):
    df = df.copy()
    m = pre_months
    pre_mask = (df['_month_idx'] < m)              # meses 1..3
    last_month_mask = (df['_month_idx'] == m)      # mes 4
    # últimas 2 semanas del mes 4 como val
    dmax = int(df.loc[last_month_mask, 'TX_DAY'].max())
    cutoff = max(1, dmax - val_days_last_month + 1)
    train_last = last_month_mask & (df['TX_DAY'] < cutoff)
    val_last   = last_month_mask & (df['TX_DAY'] >= cutoff)
    train_mask = pre_mask | train_last
    val_mask   = val_last
    return train_mask, val_mask


In [30]:
class NoOpScaler:
    def fit(self, X): return self
    def fit_transform(self, X): return X
    def transform(self, X): return X

In [32]:
def _shade_bimonth_segments(ax, x_vals, bimonth_markers, start_cycle_index=0):
    """
    Pinta franjas desde el inicio (x_min) hasta el final (x_max),
    alternando 3 colores por cada bimestre. Colorea también el 1er tramo.
    """
    import numpy as np
    if not bimonth_markers:
        return
    x_min = float(np.nanmin(x_vals))
    x_max = float(np.nanmax(x_vals))
    bounds = [x_min] + sorted([wk for wk, _ in bimonth_markers]) + [x_max + 1e-9]

    cycle = ["#9fbff2", "#f6e296", "#b7f7bb"]  # azul claro, ámbar, verde claro
    cidx = start_cycle_index % len(cycle)

    for i in range(len(bounds) - 1):
        left, right = bounds[i], bounds[i + 1]
        ax.axvspan(left, right, facecolor=cycle[cidx], alpha=0.22, linewidth=0)
        cidx = (cidx + 1) % len(cycle)

In [ ]:
def compute_bimonth_markers(eval_df: pd.DataFrame, pre_months: int, base_week: int):
  
    fy = int(eval_df['TX_YEAR'].min())
    eval_df = eval_df.copy()
    eval_df['_midx'] = (eval_df['TX_YEAR'] - fy) * 12 + eval_df['TX_MONTH'] 

    start_m = int(eval_df['_midx'].min()) 
    end_m   = int(eval_df['_midx'].max())

    M0 = start_m
    boundaries = list(range(M0 + 2, end_m + 1, 2))

    marks = []
    for b in boundaries:
        y = fy + (b - 1) // 12
        m = (b - 1) % 12 + 1
        # 1er día (en eval_df) de ese mes b
        rows = eval_df[(eval_df['TX_YEAR']==y) & (eval_df['TX_MONTH']==m)]
        if rows.empty:
            continue
        day_min = int(rows['TX_TIME_DAYS'].min())
        wk_abs  = day_min // 7 + 1
        wk_rel  = int(wk_abs - base_week + 1)
        marks.append((wk_rel, f"M{m:02d}"))
    return marks


In [ ]:


def plot_series(metrics_df, train_times_m, out_dir, month_markers=None, bimonth_markers=None,title=None,seg_plateaux_df=None):
    x = metrics_df['pos']

    def _apply_month_ticks():
        if month_markers:
            for wk, _ in month_markers:
                plt.axvline(wk, color='gray', alpha=0.15)  # líneas suaves por mes
            xs, labs = zip(*month_markers)
            plt.xticks(xs, labs, rotation=45, ha='right')

    def _apply_bimonth_lines():
        if bimonth_markers:
            for wk, _ in bimonth_markers:
                plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)  # punteada


     # === AUPRC semanal ===
    '''
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['AUPRC'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('AUPRC'); plt.title(f'AUPRC por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'auprc_semana_{title}.png')); plt.close()
    '''
 
    # === F1 semanal (umbral fijo) ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x,bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['F1'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('F1 (umbral fijo)'); plt.title(f'F1 por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'f1_semana_{title}.png')); plt.close()

        
    # === G-Mean semanal (umbral fijo) ===
    '''
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['G-Mean'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('G-Mean (umbral fijo)')
    plt.title(f'G-Mean por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'gmean_semana_{title}.png')); plt.close()
    '''
    # === Recall semanal (umbral fijo) ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['Recall'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('Recall (umbral fijo)')
    plt.title(f'Recall por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'recall_semana_{title}.png')); plt.close()

     # === Recall semanal (umbral fijo) ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['Precision'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('Precision (umbral fijo)')
    plt.title(f'Precision por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'precision_semana_{title}.png')); plt.close()

     # === Latencia semanal ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers,start_cycle_index=0)
    plt.plot(x, metrics_df['infer_ms_per_tx'].values)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('ms por chunck semanal'); plt.title(f'Latencia de inferencia ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'latencia_semana_{title}.png')); plt.close()

    if train_times_m is not None and not train_times_m.empty:
        # === Tiempo de entrenamiento semanal ===
        plt.figure()
        ax = plt.gca()
        _shade_bimonth_segments(ax, x, bimonth_markers,start_cycle_index=0)
        plt.plot(x, train_times_m['train_time_s'].values)
        _apply_month_ticks(); _apply_bimonth_lines()
        plt.xlabel('Semanas'); plt.ylabel('Tiempo de entrenamiento (s)'); plt.title(f'Tiempo de entrenamiento ({title})')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'entrenamiento_semana_{title}.png')); plt.close()



In [38]:
def compute_month_markers(eval_df: pd.DataFrame, base_week: int):
    marks = []
    months = (eval_df[['TX_YEAR','TX_MONTH']]
              .drop_duplicates()
              .sort_values(['TX_YEAR','TX_MONTH']))
    for _, r in months.iterrows():
        y, m = int(r['TX_YEAR']), int(r['TX_MONTH'])
        day_min = int(eval_df[(eval_df['TX_YEAR']==y) & (eval_df['TX_MONTH']==m)]['TX_TIME_DAYS'].min())
        wk_abs  = day_min // 7 + 1
        wk_rel  = int(wk_abs - base_week + 1)
        marks.append((wk_rel, f"{y}-{m:02d}"))
    return marks


In [ ]:
def get_scores_any(clf, X, best_iter=None):

    if hasattr(clf, "predict_proba"):
        if best_iter is not None:
           
            try:
                s = clf.predict_proba(X, iteration_range=(0, int(best_iter) + 1))[:, 1]
                return s.astype("float64"), "proba"
            except TypeError:
                try:
                    s = clf.predict_proba(X, ntree_limit=int(best_iter) + 1)[:, 1]
                    return s.astype("float64"), "proba"
                except TypeError:
                    pass
        s = clf.predict_proba(X)[:, 1]
        return s.astype("float64"), "proba"

    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        if getattr(s, "ndim", 1) > 1 and s.shape[1] == 2:
            s = s[:, 1]
        return s.astype("float64"), "decision"

    s = clf.predict(X).astype("float64")
    return s, "label"


In [40]:
def evaluate_stream(df, features, scaler, clf, thr, granularity='week', base_week=None):
    import time, numpy as np, pandas as pd
    rows = []
    tot_inf_s = 0.0
    tot_n = 0
    print("Entro al inicio de evaluate_stream")

    # Usa best_iter sólo si existe y es >= 0 (típico de XGBoost con early stopping)
    best_iter = getattr(clf, "_best_iter_internal", None)
    use_best = isinstance(best_iter, (int, np.integer)) and best_iter >= 0
    best_iter_arg = int(best_iter) if use_best else None
    
    for key, chunk, label in group_chunks(df, granularity=granularity, start_date_str=START_DATE):
        # pos relativo para el eje X
        if isinstance(key, int) and base_week is not None:
            pos = int(key) - int(base_week) + 1
        else:
            pos = int(key) if isinstance(key, int) else None

        print(f"Procesando chunk: {label}, Posición: {pos}")

        X = scaler.transform(chunk[features].astype('float32').values)
        y = chunk[TARGET].astype('int32').values
        n = int(len(y))

        # ===== Scores + tiempo (sin asumir predict_proba) =====
        t0 = time.perf_counter()
        s, _mode = get_scores_any(clf, X, best_iter=best_iter_arg)
        dt_s = time.perf_counter() - t0

        # ===== Métricas de desempeño =====
        try:
            ap = float(average_precision_score(y, s)) if y.sum() > 0 else float('nan')
        except Exception:
            ap = float('nan')

        y_pred_bin = (s >= thr).astype(int)
        f1 = float(f1_score(y, y_pred_bin, zero_division=0))

        # Sensibilidad/Especificidad y G-Mean
        sensibilidad = np.nan
        especificidad = np.nan
        g_mean = np.nan
        try:
            sensibilidad = float(recall_score(y, y_pred_bin, pos_label=1, zero_division=0))
            especificidad = float(recall_score(y, y_pred_bin, pos_label=0, zero_division=0))
            presicion = float(precision_score(y, y_pred_bin, pos_label=1, zero_division=0))
            g_mean = float(np.sqrt(sensibilidad * especificidad))
        except Exception:
            pass  # deja NaN si el chunk es degenerado

        # ===== Latencias =====
        tot_inf_s += dt_s
        tot_n     += n
        infer_total_ms_ch  = dt_s * 1e3
        infer_ms_per_tx    = (dt_s / max(1, n)) * 1e3
        infer_total_ms_cum = tot_inf_s * 1e3
        infer_ms_per_tx_c  = (tot_inf_s / max(1, tot_n)) * 1e3

        rows.append({
            "pos": pos, "label": label, "n": n,
            "AUPRC": ap, "F1": f1, "G-Mean": g_mean, "Recall": sensibilidad,"Presicion":presicion,
            #"infer_ms_per_tx": infer_ms_per_tx,                  # promedio semanal
            "infer_ms_per_tx": infer_total_ms_ch          # total semanal
           # "infer_total_ms_cum": infer_total_ms_cum,            # acumulado
            #"infer_ms_per_tx_cum": infer_ms_per_tx_c             # promedio acumulado
        })

    metrics = pd.DataFrame(rows).sort_values('pos').reset_index(drop=True)
    return metrics


In [ ]:
schedule_ym = [
  # ======================
  # PRETRAIN (Ene–Abr 2025): S1 + S2 simultáneos
  # ======================
  {"scenario": 1, "start": {"year": 2025, "month": 1}, "end": {"year": 2025, "month": 4},
   "params": {"amount_threshold": 140}},

  {"scenario": 2, "start": {"year": 2025, "month": 1}, "end": {"year": 2025, "month": 4},
   "params": {"n_per_day": 4, "window_days": 60}},

  # ======================
  # Bimestres (S1 → S2 → S3) hasta 24 meses
  # ======================

  # Bloque 1 (May–Jun 2025): S1
  {"scenario": 1, "start": {"year": 2025, "month": 5}, "end": {"year": 2025, "month": 6},
   "params": {"amount_threshold": 120}},

  # Bloque 2 (Jul–Ago 2025): S2
  {"scenario": 2, "start": {"year": 2025, "month": 7}, "end": {"year": 2025, "month": 8},
   "params": {"n_per_day": 4, "window_days": 62}},

  # Bloque 3 (Sep 2025): S3 (más suave)
  {"scenario": 3, "start": {"year": 2025, "month": 9}, "end": {"year": 2025, "month": 10},
   "params": {"n_customers_per_day": 15, "window_days": 60, "amp_factor": 6, "frac_to_flip": 1/2}},

  # Bloque 4 (Nov–Dic 2025, NAVIDAD): S1 “suavizado”
  {"scenario": 1, "start": {"year": 2025, "month": 11}, "end": {"year": 2025, "month": 12},
   "params": {"amount_threshold": 150}},

  # Bloque 5 (Ene–Feb 2026): S2
  {"scenario": 2, "start": {"year": 2026, "month": 1}, "end": {"year": 2026, "month": 2},
   "params": {"n_per_day": 4, "window_days": 59}},

  # Bloque 6 (Mar–Abr 2026): S3
  {"scenario": 3, "start": {"year": 2026, "month": 3}, "end": {"year": 2026, "month": 4},
   "params": {"n_customers_per_day": 10, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/2}},

  # Bloque 7 (May–Jun 2026): S1
  {"scenario": 1, "start": {"year": 2026, "month": 5}, "end": {"year": 2026, "month": 6},
   "params": {"amount_threshold": 120}},

  # Bloque 8 (Jul–Ago 2026): S2
  {"scenario": 2, "start": {"year": 2026, "month": 7}, "end": {"year": 2026, "month": 8},
   "params": {"n_per_day": 4, "window_days": 62}},

  # Bloque 9 (Sep–Oct 2026): S3
  {"scenario": 3, "start": {"year": 2026, "month": 9}, "end": {"year": 2026, "month": 10},
   "params": {"n_customers_per_day": 15, "window_days": 60, "amp_factor": 6, "frac_to_flip": 1/2}},

  # Bloque 10 (Nov–Dic 2026, NAVIDAD): S1 “suavizado”
  {"scenario": 1, "start": {"year": 2026, "month": 11}, "end": {"year": 2026, "month": 12},
   "params": {"amount_threshold": 150}},
]

In [ ]:
# =========================
# 1) Segmentos con scenario
# =========================
def schedule_to_week_segments_with_scenario(schedule_ym, eval_df, base_week):
    segs = []
    for step in schedule_ym:
        y1,m1 = int(step['start']['year']), int(step['start']['month'])
        y2,m2 = int(step['end']['year']),   int(step['end']['month'])
        scen  = int(step['scenario'])

        in_range = eval_df[
            ((eval_df['TX_YEAR'] >  y1) | ((eval_df['TX_YEAR']==y1) & (eval_df['TX_MONTH']>=m1))) &
            ((eval_df['TX_YEAR'] <  y2) | ((eval_df['TX_YEAR']==y2) & (eval_df['TX_MONTH']<=m2)))
        ]
        if in_range.empty:
            continue

        dmin, dmax = int(in_range['TX_TIME_DAYS'].min()), int(in_range['TX_TIME_DAYS'].max())
        wk1_abs, wk2_abs = dmin//7 + 1, dmax//7 + 1
        wk1 = int(wk1_abs - base_week + 1)
        wk2 = int(wk2_abs - base_week + 1)
        lab = f"{y1}-{m1:02d}..{y2}-{m2:02d} (S{scen})"
        segs.append({'start_pos': wk1, 'end_pos': wk2, 'label': lab, 'scenario': scen})

    return sorted(segs, key=lambda s: (s['start_pos'], s['end_pos']))


# ============================================
# 2) Plateau local (cola del segmento, robusto)
# ============================================
def _calc_plateau_from_tail(ap, window='auto', method='percentile', q=0.80):
    """
    Plateau = estadístico de cola de la serie AUPRC del segmento.
    window: entero o 'auto' (~30% del largo, min 3)
    method: 'mean' | 'median' | 'percentile'
    q: percentil (si method='percentile'), p.ej. 0.80
    """
    import numpy as np
    L = len(ap)
    W = max(3, int(round(0.3*L))) if (isinstance(window, str) and window.lower()=='auto') else int(window)
    W = max(1, min(W, L))
    tail = np.asarray(ap[-W:], dtype=float)

    if method == 'median':
        plateau = float(np.nanmedian(tail))
    elif method == 'percentile':
        plateau = float(np.nanpercentile(tail, q*100.0))
    else:
        plateau = float(np.nanmean(tail))
    return plateau, W


def compute_segment_plateaux(metrics_df, segments, window='auto', method='percentile', q=0.80):
    """
    Devuelve DF: [start_pos, end_pos, label, scenario, len_weeks, plateau_local]
    """
    import pandas as pd, numpy as np
    m = metrics_df.set_index('pos').sort_index()
    rows = []
    for seg in segments:
        s, e = int(seg['start_pos']), int(seg['end_pos'])
        sub = m.loc[m.index.intersection(range(s, e+1))]
        if sub.empty:
            rows.append({**seg, 'len_weeks': 0, 'plateau_local': np.nan})
            continue
        ap = sub['F1'].values
        plateau, _ = _calc_plateau_from_tail(ap, window=window, method=method, q=q)
        rows.append({**seg, 'len_weeks': int(len(ap)), 'plateau_local': float(plateau)})
    import pandas as pd
    return pd.DataFrame(rows).sort_values('start_pos').reset_index(drop=True)


In [ ]:
def add_terminal_rolling_features_no_datetime(df):

    import numpy as np
    import pandas as pd
    
    df = df.copy()

    df.sort_values(
        ['TERMINAL_ID', 'TX_YEAR', 'TX_MONTH', 'TX_DAY', 'TX_TIME_SECONDS'],
        kind='mergesort', inplace=True
    )

    g = df.groupby('TERMINAL_ID', sort=False)

    df['term_tx_cum'] = g.cumcount().astype('int32') + 1

    term_cum_frd = g['TX_FRAUD'].cumsum()
    df['term_frd_cum'] = term_cum_frd.groupby(df['TERMINAL_ID']).shift(1).fillna(0).astype('float32')
 
    prior = float(df['TX_FRAUD'].mean())  # prevalencia global aprox.
    k = 20.0                              # fuerza de suavizado
    denom = (df['term_tx_cum'] - 1 + k).astype('float32')  # -1 porque es hasta t-1
    df['term_fraud_rate_cum'] = ((df['term_frd_cum'] + k*prior) / denom).astype('float32')

    df['term_tx_ewm'] = g['TX_FRAUD'].apply(
        lambda s: s.shift(1).ewm(alpha=0.2, adjust=False).mean()
    ).reset_index(level=0, drop=True).astype('float32')

    df['term_time_since_prev'] = g['TX_TIME_SECONDS'].diff().fillna(1e9).astype('float32')

    df.sort_values(
        ['TX_YEAR','TX_MONTH','TX_DAY','TX_TIME_SECONDS'],
        kind='mergesort', inplace=True
    )
    df.reset_index(drop=True, inplace=True)

    return df


In [ ]:
# ========= Helpers para el baseline rolling =========

def week_to_month_map(df):
    m = df.groupby('_week_idx')['_month_idx'].max().astype(int)
    return m.to_dict()

def build_window_masks(df, up_to_week, months_back=4, week2month=None):

    if week2month is None:
        week2month = week_to_month_map(df)

    weeks_in_window = int(months_back * (52 / 12)) 
    last_week_in_window = int(up_to_week)
    first_week_in_window = max(1, last_week_in_window - weeks_in_window + 1)

   # print(f"up_to_week: {up_to_week} (Window size: {weeks_in_window} weeks)")
   # print(f"first_week: {first_week_in_window}, last_week: {last_week_in_window}")

    in_weeks = (df['_week_idx'] >= first_week_in_window) & (df['_week_idx'] <= last_week_in_window)

    # Validación = exactamente la semana 'up_to_week'
    val_mask   = in_weeks & (df['_week_idx'] == int(up_to_week))
    
    # Train = resto de la ventana (pasado estricto)
    train_mask = in_weeks & (df['_week_idx'] <  int(up_to_week))

    return train_mask, val_mask

def fit_window_model(window_train_df, window_val_df, features, model_name,
                     target='TX_FRAUD', prev_thr=None, recalibrate_thr=False, random_state=42):
    import time
    # Datos
    Xtr = window_train_df[features].astype('float32').values
    ytr = window_train_df[target].astype('int32').values

    # Scaler/modelo según tipo
    name = model_name.lower()
    if name in ("svm","svc","lr","logreg","logistic","logistic_regression"):
        scaler = StandardScaler().fit(Xtr)
        Xtr_s  = scaler.transform(Xtr)
    else:
        scaler = NoOpScaler()
        Xtr_s  = Xtr

    clf = make_model(model_name, scale_pos_weight=float((len(ytr)-ytr.sum())/max(1,ytr.sum())), random_state=random_state)

 
    t0 = time.perf_counter()
    clf.fit(Xtr_s, ytr)
    train_time_s = time.perf_counter() - t0

 
    thr = prev_thr if prev_thr is not None else 0.5

    ms_per_tx_val = np.nan
    if recalibrate_thr and window_val_df is not None and len(window_val_df) > 0:
        Xv = window_val_df[features].astype('float32').values
        yv = window_val_df[target].astype('int32').values
        Xv_s = scaler.transform(Xv)

        t0 = time.perf_counter()
        s, _mode = get_scores(clf, Xv_s)  # usa proba si hay, si no decision_function
        dt = time.perf_counter() - t0
        ms_per_tx_val = (dt / max(1, len(yv))) * 1e3

        prec, rec, thr_grid = precision_recall_curve(yv, s)
        f1s = (2*prec*rec)/(prec+rec+1e-12)
        if len(thr_grid) > 0:
            best_idx = int(np.nanargmax(f1s[:-1]))
            thr = float(thr_grid[best_idx])

    return scaler, clf, thr, train_time_s, ms_per_tx_val

def rolling_retrain_baseline(df_total, train_df, val_df, eval_df, features,
                             model_name, base_week, months_window=4,
                             recalibrate_thr=False, random_state=42):

    scaler, clf, thr, ap_ref, f1_ref, train_time_init, ms_per_tx_val_init = fit_model(
        train_df, val_df, features, model_name=model_name, random_state=random_state
    )

    rows = []
    train_times = [] 
    week2month = week_to_month_map(df_total)

    #print("week2month:", week2month)


    for key, chunk, label in group_chunks(eval_df, granularity='week', start_date_str=START_DATE):
        # semana relativa para gráficos
        pos = (int(key) - int(base_week) + 1) if base_week is not None else int(key)
        print('Entrenando la semana N:', pos)
        # ======= (a) EVALUAR con el modelo actual en esta semana =======
        X = scaler.transform(chunk[features].astype('float32').values)
        y = chunk['TX_FRAUD'].astype('int32').values
        n = int(len(y))

        t0 = time.perf_counter()
        s, _mode = get_scores(clf, X)
        dt_s = time.perf_counter() - t0

        try:
            ap = float(average_precision_score(y, s)) if y.sum() > 0 else float('nan')
        except Exception:
            ap = float('nan')
        f1 = float(f1_score(y, (s >= thr).astype(int), zero_division=0))

        y_pred_bin = (s >= thr).astype(int)
        try:
       
            sensibilidad = float(recall_score(y, y_pred_bin, pos_label=1, zero_division=0))
         
            especificidad = float(recall_score(y, y_pred_bin, pos_label=0, zero_division=0))
            presicion = float(precision_score(y, y_pred_bin, pos_label=1, zero_division=0))
            
            # Calcular G-Mean
            g_mean = float(np.sqrt(sensibilidad * especificidad))
        except Exception:
            g_mean = float('nan')

        rows.append({
            "pos": pos, "label": label, "n": n,
            "AUPRC": ap, "F1": f1, "G-Mean": g_mean, "Recall": sensibilidad,"Presicion":presicion,
            #"infer_ms_per_tx": (dt_s / max(1, n)) * 1e3,
            "infer_ms_per_tx": dt_s * 1e3
        })

 
        train_mask, val_mask = build_window_masks(df_total, up_to_week=int(key),
                                                  months_back=months_window, week2month=week2month)
        win_train = df_total[train_mask]
        win_val   = df_total[val_mask] if recalibrate_thr else None

        start_date = pd.Timestamp(
            year=win_train['TX_YEAR'].iloc[0],
            month=win_train['TX_MONTH'].iloc[0],
            day=win_train['TX_DAY'].iloc[0]
        ).strftime('%d-%m-%Y')

        end_date = pd.Timestamp(
            year=win_train['TX_YEAR'].iloc[-1],
            month=win_train['TX_MONTH'].iloc[-1],
            day=win_train['TX_DAY'].iloc[-1]
        ).strftime('%d-%m-%Y')

        #print("Inicio mascara: ", start_date)
        #print("Fin mascara: ", end_date)

        scaler, clf, thr, train_time_s, _ms_per_tx_val = fit_window_model(
            win_train, win_val, features, model_name,
            prev_thr=thr, recalibrate_thr=recalibrate_thr, random_state=random_state
        )
        train_times.append({"pos": pos, "train_time_s": train_time_s})


    metrics = pd.DataFrame(rows).sort_values('pos').reset_index(drop=True)
    train_df_times = pd.DataFrame(train_times)
    return metrics, train_df_times, {"ap_ref": ap_ref, "f1_ref": f1_ref, "thr0": thr, "train_time_init": train_time_init}


In [ ]:
df = load_all_parquet_by_year(BASE_PATH)
df = add_time_indexes(df)

df = add_terminal_rolling_features_no_datetime(df)

cols_quitar = ['CUSTOMER_ID', 'TERMINAL_ID', 'TX_TIME_SECONDS']
df = df.drop(columns=[c for c in cols_quitar if c in df.columns])


EXTRA_FEATS = [
    'term_tx_cum', 'term_frd_cum', 'term_fraud_rate_cum',
    'term_tx_ewm', 'term_time_since_prev'
]
features = [c for c in (FEATURES + EXTRA_FEATS) if c in df.columns]


train_mask, val_mask = split_pretrain_val(df, pre_months=PRE_MONTHS, val_days_last_month=VAL_DAYS_LAST_MONTH)
train_df = df[train_mask]
val_df   = df[val_mask]

print("Entrenando baseline (LR) con meses 1..4...")
features = [c for c in FEATURES if c in df.columns]

results = {}

print("Evaluando serie temporal...")
eval_df = df[df['_month_idx'] >= (PRE_MONTHS + 1)]
base_week = compute_base_week(eval_df)   
print("DF de evaluacion\n")
print(eval_df.columns.to_list())

print(eval_df.head())

month_marks = compute_month_markers(eval_df,base_week)
bimonth_marks  = compute_bimonth_markers(eval_df, pre_months=PRE_MONTHS, base_week=base_week)
segments1 = schedule_to_week_segments_with_scenario(schedule_ym, eval_df, base_week)


Entrenando baseline (LR) con meses 1..4...
Evaluando serie temporal...
Semana base: 18, Fecha: 2025-05-01
DF de evaluacion

['TX_YEAR', 'TX_MONTH', 'TX_DAY', 'TX_TIME_DAYS', 'TX_AMOUNT', 'x_customer_id', 'y_customer_id', 'mean_amount', 'std_amount', 'mean_nb_tx_per_day', 'x_terminal_id', 'y_terminal_id', 'TX_FRAUD', '_month_idx', '_week_idx', '_day_abs', 'term_tx_cum', 'term_frd_cum', 'term_fraud_rate_cum', 'term_tx_ewm', 'term_time_since_prev']
        TX_YEAR  TX_MONTH  TX_DAY  TX_TIME_DAYS  TX_AMOUNT  x_customer_id  \
344257     2025         5       1           120  10.910000      54.367805   
344258     2025         5       1           120  67.220001      50.837330   
344259     2025         5       1           120  66.949997      77.480385   
344260     2025         5       1           120   6.320000      65.135933   
344261     2025         5       1           120  56.380001      69.957504   

        y_customer_id  mean_amount  std_amount  mean_nb_tx_per_day  ...  \
344257      

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

def calculate_volatility_by_segment(metrics_df, seg_plateaux_df):
   
    metrics_indexed = metrics_df.set_index('pos')
    results = []
    
    for _, segment in seg_plateaux_df.iterrows():
        s, e, label = int(segment['start_pos']), int(segment['end_pos']), segment['label']
        
        f1_series = metrics_indexed.loc[s:e]['F1'].dropna()
        
        if len(f1_series) < 2:
        
            volatility = np.nan 
        else:
            mean_val = f1_series.mean()
            std_val = f1_series.std()
            if mean_val > 0.001:
                volatility = std_val / mean_val
            else:
                volatility = 0.0
            
        results.append({'label': label, 'volatility': volatility})
        
    return pd.DataFrame(results)

def plot_volatility_chart(volatility_df, out_dir, title):
    
    if volatility_df.empty:
        return
        
    plt.figure(figsize=(10, 6)) 
    
    plt.bar(volatility_df['label'], volatility_df['volatility'])
    
    plt.title(f"Volatilidad del F1 por Patrón ({title})")
    plt.ylabel("Volatilidad (Desv. Estándar F1)")
    plt.xlabel("Segmento de Patrón")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'volatilidad_f1_{title}.png'))
    plt.close()

def calculate_stable_recovery_by_segment(metrics_df, seg_plateaux_df, 
                                           recovery_threshold_pct=0.9, 
                                           stability_window_k=3):

    metrics_indexed = metrics_df.set_index('pos')
    results = []

    for _, segment in seg_plateaux_df.iterrows():
        s, e, label = int(segment['start_pos']), int(segment['end_pos']), segment['label']
        f1_plateau = segment['plateau_local'] 
        
        recovery_weeks = np.nan 
        
        if not np.isfinite(f1_plateau) or f1_plateau <= 0:
            results.append({'label': label, 'recovery_weeks': recovery_weeks})
            continue
            
        f1_threshold = f1_plateau * recovery_threshold_pct
        f1_series = metrics_indexed.loc[s:e]['F1'].values # F1s solo de este segmento
        
        if len(f1_series) < stability_window_k:
            results.append({'label': label, 'recovery_weeks': recovery_weeks})
            continue


        for i in range(len(f1_series) - stability_window_k + 1):
            window = f1_series[i : i + stability_window_k]
            
            if np.all(window >= f1_threshold):
                recovery_weeks = i 
                break
        
        results.append({'label': label, 'recovery_weeks': recovery_weeks})
        
    return pd.DataFrame(results)

def plot_recovery_chart(recovery_df, out_dir, title, stability_window_k=3):

    if recovery_df.empty:
        return
        
    plt.figure(figsize=(10, 6))
    
    plot_df = recovery_df.copy()
    
  
    max_real_recovery = plot_df['recovery_weeks'].max()
    if not np.isfinite(max_real_recovery) or max_real_recovery == 0:
        max_real_recovery = 8

    failure_value = max_real_recovery * 1.1 + 1 
    
    plot_df['plot_value'] = plot_df['recovery_weeks'].fillna(failure_value)
    

    plot_df['plot_value'] = plot_df['plot_value'].apply(lambda x: 0.1 if x == 0 else x)

    bars = plt.bar(plot_df['label'], plot_df['plot_value'])

    legend_labels = {}
    
    for i, bar in enumerate(bars):
        valor_real = plot_df['recovery_weeks'].iloc[i]
        
        if not np.isfinite(valor_real):
            bar.set_color('red')
            bar.set_edgecolor('black')
            legend_labels['Nunca se recuperó'] = bar
            
        elif valor_real == 0:
            bar.set_color('green')
            legend_labels['Éxito Instantáneo'] = bar
            
        else:
            legend_labels['Recuperación Exitosa'] = bar

    plt.title(f"Tiempo de Recuperación Estable (K={stability_window_k}) ({title})")
    plt.ylabel(f"Semanas para Recuperación (K={stability_window_k})")
    plt.xlabel("Segmento de Patrón")
    plt.xticks(rotation=45, ha='right')
    
 
    if legend_labels:
         plt.legend(legend_labels.values(), legend_labels.keys())

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'recuperacion_f1_{title}.png'))
    plt.close()

In [ ]:

model_names = ["svm", "rf", "logreg","xgb"] #,"svm","xgb" ,"rf","logreg"

for mname in model_names:
    print(f"\n=== Entrenando {mname.upper()} con meses 1..4 ===")
    scaler, clf, thr, ap_ref, f1_ref, train_time_s, mspt_val = fit_model(train_df, val_df, features, mname)
    print(f"Val AUPRC={ap_ref:.4f} | Val F1={f1_ref:.4f} @thr={thr:.4f} | train_s={train_time_s:.2f} | ms/tx_val={mspt_val:.3f}")
    

    metrics_m = evaluate_stream(eval_df, features, scaler, clf, thr, granularity='week', base_week=base_week)
    results[mname] = {
        "scaler": scaler, "clf": clf, "thr": thr,
        "ap_ref": ap_ref, "f1_ref": f1_ref,
        "train_time_s": train_time_s, "mspt_val": mspt_val,
        "metrics": metrics_m
    }

    model_dir = os.path.join(OUT_DIR, mname.upper()) 
    os.makedirs(model_dir, exist_ok=True)

    metrics_m.to_csv(os.path.join(model_dir, 'weekly_metrics.csv'), index=False)

  
    plot_series(metrics_m,None, model_dir, month_markers=month_marks, bimonth_markers=bimonth_marks,title=f"BASELINE 1 {mname.upper()}")

    seg_plateaux = compute_segment_plateaux(
        metrics_m, segments1, window=8, method='percentile', q=0.80
    )


    print(f"Calculando métricas de adaptación para {mname.upper()}...")
    

    # (Usa el 'seg_plateaux' que ya calculaste)
    volatility_df = calculate_volatility_by_segment(metrics_m, seg_plateaux)
    
    # Guardamos los datos y el gráfico
    #volatility_df.to_csv(os.path.join(model_dir, 'volatilidad_por_patron.csv'), index=False)
    plot_volatility_chart(volatility_df, model_dir, title=f"BASELINE 1 {mname.upper()}")

    K_ESTABILIDAD =5  # Puedes ajustar esto (ej. 2 o 3 semanas)
    PCT_RECUPERACION = 1
    
    recovery_df = calculate_stable_recovery_by_segment(
        metrics_m, 
        seg_plateaux,
        recovery_threshold_pct=PCT_RECUPERACION,
        stability_window_k=K_ESTABILIDAD
    )
    
    #recovery_df.to_csv(os.path.join(model_dir, 'recuperacion_por_patron.csv'), index=False)
    plot_recovery_chart(recovery_df, model_dir, 
                        title=f"BASELINE 1 {mname.upper()}", 
                        stability_window_k=K_ESTABILIDAD)


    # Resumen
print("\n=== RESUMEN BASELINE #1 ===")
print(f"AUPRC de referencia (val mes 4): {ap_ref:.4f}")
print(f"F1 de referencia (val mes 4)   : {f1_ref:.4f} @thr={thr:.3f}")
print(f"Tiempo de entrenamiento (s)     : {train_time_s:.2f}")

print(f"\nArchivos guardados en: {OUT_DIR}")


=== Entrenando SVM con meses 1..4 ===
Val AUPRC=0.9358 | Val F1=0.8807 @thr=0.4901 | train_s=0.58 | ms/tx_val=0.000
Entro al inicio de evaluate_stream
Procesando chunk: 2025-W18, Posición: 1
Procesando chunk: 2025-W19, Posición: 2
Procesando chunk: 2025-W20, Posición: 3
Procesando chunk: 2025-W21, Posición: 4
Procesando chunk: 2025-W22, Posición: 5
Procesando chunk: 2025-W23, Posición: 6
Procesando chunk: 2025-W24, Posición: 7
Procesando chunk: 2025-W25, Posición: 8
Procesando chunk: 2025-W26, Posición: 9
Procesando chunk: 2025-W27, Posición: 10
Procesando chunk: 2025-W28, Posición: 11
Procesando chunk: 2025-W29, Posición: 12
Procesando chunk: 2025-W30, Posición: 13
Procesando chunk: 2025-W31, Posición: 14
Procesando chunk: 2025-W32, Posición: 15
Procesando chunk: 2025-W33, Posición: 16
Procesando chunk: 2025-W34, Posición: 17
Procesando chunk: 2025-W35, Posición: 18
Procesando chunk: 2025-W36, Posición: 19
Procesando chunk: 2025-W37, Posición: 20
Procesando chunk: 2025-W38, Posición:

In [ ]:

model_names = ["rf","svm", "logreg","xgb" ] 

for mname in model_names:
    print(f"\n=== Rolling retrain {mname.upper()} (ventana 4 meses) ===")
    metrics_m, train_times_m, seed_info = rolling_retrain_baseline(
        df_total=df,               
        train_df=train_df,         # meses 1..4 (pretraining)
        val_df=val_df,             # calibración inicial
        eval_df=eval_df,         
        features=features,
        model_name=mname,
        base_week=base_week,
        months_window=25,        # ventana de 4 meses = 16 semanas (aprox) o 25 para ventana incremental
        recalibrate_thr=False     
    )

    # carpeta por modelo
    model_dir = os.path.join(OUT_DIR, f"{mname.upper()}_BASELINE3")
    os.makedirs(model_dir, exist_ok=True)

    # guarda métricas y tiempos
    metrics_m.to_csv(os.path.join(model_dir, "weekly_metrics.csv"), index=False)
    #train_times_m.to_csv(os.path.join(model_dir, "train_times.csv"), index=False)

    # gráficas estándar
    plot_series(metrics_m, train_times_m, model_dir, month_markers=month_marks, bimonth_markers=bimonth_marks,title=f"BASELINE 3 {mname.upper()}")

    segments_ym = schedule_to_week_segments_with_scenario(schedule_ym, eval_df, base_week)
    seg_plateaux = compute_segment_plateaux(metrics_m, segments_ym, window=8, method='percentile', q=0.80)

    print(f"Calculando métricas de adaptación para {mname.upper()}...")
    
    # --- 1. Métrica de Volatilidad ---
    volatility_df = calculate_volatility_by_segment(metrics_m, seg_plateaux)
    
    #volatility_df.to_csv(os.path.join(model_dir, 'volatilidad_por_patron.csv'), index=False)
    plot_volatility_chart(volatility_df, model_dir, title=f"BASELINE 3 {mname.upper()}")

    # --- 2. Métrica de Recuperación Estable ---
    K_ESTABILIDAD =5  
    PCT_RECUPERACION = 1
    
    recovery_df = calculate_stable_recovery_by_segment(
        metrics_m, 
        seg_plateaux,
        recovery_threshold_pct=PCT_RECUPERACION,
        stability_window_k=K_ESTABILIDAD
    )
    # Guardamos los datos y el gráfico
    #recovery_df.to_csv(os.path.join(model_dir, 'recuperacion_por_patron.csv'), index=False)
    plot_recovery_chart(recovery_df, model_dir, 
                        title=f"BASELINE 3 {mname.upper()}", 
                        stability_window_k=K_ESTABILIDAD)


print("\nListo: baseline rolling (4m) generado para todos los modelos")


=== Rolling retrain RF (ventana 4 meses) ===
Entrenando la semana N: 1
Entrenando la semana N: 2
Entrenando la semana N: 3
Entrenando la semana N: 4
Entrenando la semana N: 5
Entrenando la semana N: 6
Entrenando la semana N: 7
Entrenando la semana N: 8
Entrenando la semana N: 9
Entrenando la semana N: 10
Entrenando la semana N: 11
Entrenando la semana N: 12
Entrenando la semana N: 13
Entrenando la semana N: 14
Entrenando la semana N: 15
Entrenando la semana N: 16
Entrenando la semana N: 17
Entrenando la semana N: 18
Entrenando la semana N: 19
Entrenando la semana N: 20
Entrenando la semana N: 21
Entrenando la semana N: 22
Entrenando la semana N: 23
Entrenando la semana N: 24
Entrenando la semana N: 25
Entrenando la semana N: 26
Entrenando la semana N: 27
Entrenando la semana N: 28
Entrenando la semana N: 29
Entrenando la semana N: 30
Entrenando la semana N: 31
Entrenando la semana N: 32
Entrenando la semana N: 33
Entrenando la semana N: 34
Entrenando la semana N: 35
Entrenando la sema